<a href="https://colab.research.google.com/github/9lana/WAY-AI/blob/main/sentiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import json

TRAINING DATA

In [ ]:


training_data = [
    ("this is amazing", 1),
    ("i love this product", 1),
    ("absolutely wonderful experience", 1),
    ("great quality and fast delivery", 1),
    ("i am very happy with this", 1),
    ("excellent work keep it up", 1),
    ("best purchase i ever made", 1),
    ("fantastic service highly recommend", 1),
    ("really good and very useful", 1),
    ("superb quality loved it", 1),
    ("this is terrible", 0),
    ("i hate this so much", 0),
    ("worst experience ever", 0),
    ("absolutely awful and disgusting", 0),
    ("very bad quality do not buy", 0),
    ("horrible service never again", 0),
    ("complete waste of money", 0),
    ("disappointing and broken on arrival", 0),
    ("really poor and useless product", 0),
    ("garbage total garbage", 0),
]

In [ ]:
def tokenize(sentence):
    return sentence.lower().split()

In [ ]:
def build_vocab(data):
    vocab = {}
    idx = 0
    for sentence, _ in data:
        for word in tokenize(sentence):
            if word not in vocab:
                vocab[word] = idx
                idx += 1
    return vocab

In [ ]:
def bag_of_words(sentence, vocab):
    vector = [0.0] * len(vocab)
    for word in tokenize(sentence):
        if word in vocab:
            vector[vocab[word]] += 1
    return vector

In [ ]:
vocab = build_vocab(training_data)
vocab_size = len(vocab)

In [ ]:
idk = []
for s, _ in training_data:
    lol = bag_of_words(s, vocab)
    idk.append(lol)
X = torch.tensor(idk, dtype=torch.float32)

In [ ]:
num = []
for _, label in training_data:
    lol = float(label)
    num.append(lol)
y1 = torch.tensor(num, dtype=torch.float32)
Y = y1.unsqueeze(1)

In [ ]:
print(f"{X}\n")
print(f"{Y}\n")

tensor([[1., 1., 1.,  ..., 0., 0., 0.],
        [1., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 1., 0., 0.],
        [0., 0., 0.,  ..., 0., 2., 1.]])

tensor([[1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.]])



In [ ]:

print(f"X shape: {X.shape}  ← 20 sentences × {vocab_size} words")
print(f"y shape: {Y.shape}  ← 20 labels\n")

X shape: torch.Size([20, 62])  ← 20 sentences × 62 words
y shape: torch.Size([20, 1])  ← 20 labels



In [ ]:
model = nn.Sequential(
    nn.Linear(vocab_size, 1),
    nn.Sigmoid()
)
print("Model layer")
for name, layer in model.named_modules():
    if name:
        print(f"{layer}")

Model layer
Linear(in_features=62, out_features=1, bias=True)
Sigmoid()


In [ ]:
criterion = nn.BCELoss()
optimiser = torch.optim.SGD(model.parameters(), lr=0.1)

In [ ]:
epochs = 300
for epoch in range(epochs):
    predicted = model(X)
    l = criterion(predicted, Y)
    optimiser.zero_grad()
    l.backward()
    optimiser.step()

    if (epoch + 1) % 50 == 0:
        correct = ((predicted >= 0.5) == (Y == 1)).float().sum().item()
        accuracy = correct / len(training_data) * 100
        print(f"Epoch {epoch+1:>3} | Loss: {l.item():.4f} | Accuracy: {accuracy:.1f}%")

Epoch  50 | Loss: 0.5310 | Accuracy: 95.0%
Epoch 100 | Loss: 0.4320 | Accuracy: 100.0%
Epoch 150 | Loss: 0.3631 | Accuracy: 100.0%
Epoch 200 | Loss: 0.3124 | Accuracy: 100.0%
Epoch 250 | Loss: 0.2736 | Accuracy: 100.0%
Epoch 300 | Loss: 0.2429 | Accuracy: 100.0%


In [ ]:
torch.save(model.state_dict(), "model.pth")
with open("vocab.json", "w") as f:
    json.dump(vocab, f)

In [ ]:
def predict(sentence):
    x = bag_of_words(sentence, vocab)
    x_tensor = torch.tensor(x, dtype=torch.float32).unsqueeze(1)
    with torch.no_grad():
        prod = model(x_tensor).item()
    if prod >= 0.5:
        label = "Good"
    else:
        label = "Bad"
    return label

In [ ]:
def predict(sentence):
    x = bag_of_words(sentence, vocab)
    x_tensor = torch.tensor(x, dtype=torch.float32).unsqueeze(0) # Corrected here
    with torch.no_grad():
        prod = model(x_tensor).item()
    if prod >= 0.5:
        label = "Good"
    else:
        label = "Bad"
    return label

while True:
    try:
        user_input = input("Text ").strip()
        label = predict(user_input)
        print(f"{label}")
    except EOFError:
        print("Exiting input loop.")
        break

Good
Good
Bad
Bad
